In [1]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()
spark = SparkSession. \
builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv026931/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
from pyspark.sql.functions import *

In [3]:
loans_repay_raw_df = spark.read \
.format("csv") \
.option("header",True) \
.option("inferSchema",True) \
.load("/user/itv026931/lendingclubproject/raw/loans_repayments_csv")

In [4]:
loans_repay_raw_df

loan_id,total_rec_prncp,total_rec_int,total_rec_late_fee,total_pymnt,last_pymnt_amnt,last_pymnt_d,next_pymnt_d
76003861,6423.8,5607.85,0.0,12031.65,574.88,Jan-2018,null
76263914,2400.0,323.18,0.0,2723.1800875923,905.18,Dec-2017,null
75537401,12600.0,1392.76,0.0,13992.7563947386,3437.74,Aug-2018,null
75038986,7805.03,7583.71,0.0,15388.74,440.72,Mar-2019,Apr-2019
76301424,2614.78,1080.09,0.0,3694.87,153.89,Apr-2018,null
75333198,8664.09,3431.32,68.8,12164.21,343.9,Mar-2019,Apr-2019
76391453,35000.0,10266.01,0.0,45266.0062708179,18884.35,Feb-2019,null
76363364,15000.0,2093.11,0.0,17093.108291696,4648.59,Aug-2018,null
76272510,14463.21,11178.79,110.13,25752.13,734.18,Mar-2019,Apr-2019
76304116,4800.0,1051.65,0.0,5851.6523334103,1761.59,Dec-2017,null


In [4]:
loan_repayments_schema = '''loan_id string,total_principal_received float,total_interest_received float,total_late_fee_received float,
total_payment_received float,last_payment_amount float,last_payment_date string,next_payment_date string'''

In [5]:
loans_repay_raw_df.printSchema()

root
 |-- loan_id: string (nullable = true)
 |-- total_rec_prncp: string (nullable = true)
 |-- total_rec_int: string (nullable = true)
 |-- total_rec_late_fee: string (nullable = true)
 |-- total_pymnt: string (nullable = true)
 |-- last_pymnt_amnt: string (nullable = true)
 |-- last_pymnt_d: string (nullable = true)
 |-- next_pymnt_d: string (nullable = true)



In [6]:
loans_repay_raw_df = spark.read \
.format("csv") \
.option("header",True) \
.schema(loan_repayments_schema) \
.load("/user/itv026931/lendingclubproject/raw/loans_repayments_csv")

Add Ingestion_date

In [7]:
loans_repay_df_ingestd = loans_repay_raw_df.withColumn("Ingest_date",current_timestamp())

In [15]:
loans_repay_df_ingestd

loan_id,total_principal_received,total_interest_received,total_late_fee_received,total_payment_received,last_payment_amount,last_payment_date,next_payment_date,Ingest_date
76003861,6423.8,5607.85,0.0,12031.65,574.88,Jan-2018,null,2026-09-28 21:58:...
76263914,2400.0,323.18,0.0,2723.1802,905.18,Dec-2017,null,2026-09-28 21:58:...
75537401,12600.0,1392.76,0.0,13992.757,3437.74,Aug-2018,null,2026-09-28 21:58:...
75038986,7805.03,7583.71,0.0,15388.74,440.72,Mar-2019,Apr-2019,2026-09-28 21:58:...
76301424,2614.78,1080.09,0.0,3694.87,153.89,Apr-2018,null,2026-09-28 21:58:...
75333198,8664.09,3431.32,68.8,12164.21,343.9,Mar-2019,Apr-2019,2026-09-28 21:58:...
76391453,35000.0,10266.01,0.0,45266.008,18884.35,Feb-2019,null,2026-09-28 21:58:...
76363364,15000.0,2093.11,0.0,17093.107,4648.59,Aug-2018,null,2026-09-28 21:58:...
76272510,14463.21,11178.79,110.13,25752.13,734.18,Mar-2019,Apr-2019,2026-09-28 21:58:...
76304116,4800.0,1051.65,0.0,5851.6523,1761.59,Dec-2017,null,2026-09-28 21:58:...


In [8]:
loans_repay_df_ingestd.createOrReplaceTempView("loan_repayments")

Finding out the null values

In [70]:
loans_repay_df_ingestd.filter("total_principal_received is null").count()

69

In [26]:
loans_repay_df_ingestd.filter(col("total_principal_received").isNull()).count()

69

In [27]:
spark.sql("select count(*) from loan_repayments where total_principal_received is null")

count(1)
69


In [9]:
columns_to_check = ["total_principal_received","total_interest_received","total_late_fee_received","total_payment_received",
                    "last_payment_amount"]

In [10]:
loans_repay_filtered = loans_repay_df_ingestd.na.drop(subset=columns_to_check)

In [43]:
loans_repay_filtered.count()

2260498

In [51]:
loans_repay_filtered.filter("total_payment_received == 0 and total_principal_received != 0")

loan_id,total_principal_received,total_interest_received,total_late_fee_received,total_payment_received,last_payment_amount,last_payment_date,next_payment_date,Ingest_date
1064185,11600.98,11600.98,10000.0,0.0,0.0,0.0,Dec-2014,2026-09-28 22:41:...
516382,21890.229,21856.03,16000.0,0.0,0.0,0.0,Mar-2014,2026-09-28 22:41:...
528899,3045.0364,3019.64,2500.0,0.0,0.0,0.0,Jan-2013,2026-09-28 22:41:...
527598,2398.9092,2220.51,2200.0,0.0,0.0,0.0,Jul-2011,2026-09-28 22:41:...
525697,21797.86,19894.9,15750.0,0.0,0.0,0.0,Jun-2015,2026-09-28 22:41:...
522641,3146.8193,3146.82,3000.0,0.0,0.0,0.0,Sep-2011,2026-09-28 22:41:...
515655,29938.576,29905.75,22800.0,0.0,0.0,0.0,May-2013,2026-09-28 22:41:...
501234,15219.313,15155.9,12000.0,0.0,0.0,0.0,May-2013,2026-09-28 22:41:...
498194,11642.714,11031.47,10000.0,0.0,0.0,0.0,Jan-2013,2026-09-28 22:41:...
495171,11138.843,10024.96,10000.0,0.0,0.0,0.0,Apr-2013,2026-09-28 22:41:...


In [11]:
loans_payments_fixed_df = loans_repay_filtered.withColumn("total_payment_received",
                                                          when((col("total_principal_received") != 0) &
                                                             (col("total_payment_received") == 0),
                               col("total_principal_received") + col("total_interest_received") + col("total_late_fee_received"))
                               .otherwise(col("total_payment_received"))
                               )

In [54]:
loans_payments_fixed_df

loan_id,total_principal_received,total_interest_received,total_late_fee_received,total_payment_received,last_payment_amount,last_payment_date,next_payment_date,Ingest_date
76003861,6423.8,5607.85,0.0,12031.65,574.88,Jan-2018,null,2026-09-28 22:45:...
76263914,2400.0,323.18,0.0,2723.1802,905.18,Dec-2017,null,2026-09-28 22:45:...
75537401,12600.0,1392.76,0.0,13992.757,3437.74,Aug-2018,null,2026-09-28 22:45:...
75038986,7805.03,7583.71,0.0,15388.74,440.72,Mar-2019,Apr-2019,2026-09-28 22:45:...
76301424,2614.78,1080.09,0.0,3694.87,153.89,Apr-2018,null,2026-09-28 22:45:...
75333198,8664.09,3431.32,68.8,12164.21,343.9,Mar-2019,Apr-2019,2026-09-28 22:45:...
76391453,35000.0,10266.01,0.0,45266.008,18884.35,Feb-2019,null,2026-09-28 22:45:...
76363364,15000.0,2093.11,0.0,17093.107,4648.59,Aug-2018,null,2026-09-28 22:45:...
76272510,14463.21,11178.79,110.13,25752.13,734.18,Mar-2019,Apr-2019,2026-09-28 22:45:...
76304116,4800.0,1051.65,0.0,5851.6523,1761.59,Dec-2017,null,2026-09-28 22:45:...


In [55]:
loans_payments_fixed_df.filter("loan_id == 516382")

loan_id,total_principal_received,total_interest_received,total_late_fee_received,total_payment_received,last_payment_amount,last_payment_date,next_payment_date,Ingest_date
516382,21890.229,21856.03,16000.0,59746.258,0.0,0.0,Mar-2014,2026-09-28 22:46:...


In [12]:
loans_payments_fixed2_df = loans_payments_fixed_df.filter("total_payment_received != 0.0")

In [86]:
loans_payments_fixed2_df

loan_id,total_principal_received,total_interest_received,total_late_fee_received,total_payment_received,last_payment_amount,last_payment_date,next_payment_date,Ingest_date
76003861,6423.8,5607.85,0.0,12031.65,574.88,Jan-2018,null,2026-09-28 23:14:...
76263914,2400.0,323.18,0.0,2723.1802,905.18,Dec-2017,null,2026-09-28 23:14:...
75537401,12600.0,1392.76,0.0,13992.757,3437.74,Aug-2018,null,2026-09-28 23:14:...
75038986,7805.03,7583.71,0.0,15388.74,440.72,Mar-2019,Apr-2019,2026-09-28 23:14:...
76301424,2614.78,1080.09,0.0,3694.87,153.89,Apr-2018,null,2026-09-28 23:14:...
75333198,8664.09,3431.32,68.8,12164.21,343.9,Mar-2019,Apr-2019,2026-09-28 23:14:...
76391453,35000.0,10266.01,0.0,45266.008,18884.35,Feb-2019,null,2026-09-28 23:14:...
76363364,15000.0,2093.11,0.0,17093.107,4648.59,Aug-2018,null,2026-09-28 23:14:...
76272510,14463.21,11178.79,110.13,25752.13,734.18,Mar-2019,Apr-2019,2026-09-28 23:14:...
76304116,4800.0,1051.65,0.0,5851.6523,1761.59,Dec-2017,null,2026-09-28 23:14:...


In [13]:
loans_payments_ldate_fixed_df = loans_payments_fixed2_df.withColumn(
"last_payment_date",
when(
(col("last_payment_date") == 0),
None
).otherwise(col("last_payment_date"))
)

In [88]:
loans_payments_ldate_fixed_df

loan_id,total_principal_received,total_interest_received,total_late_fee_received,total_payment_received,last_payment_amount,last_payment_date,next_payment_date,Ingest_date
76003861,6423.8,5607.85,0.0,12031.65,574.88,Jan-2018,null,2026-09-28 23:14:...
76263914,2400.0,323.18,0.0,2723.1802,905.18,Dec-2017,null,2026-09-28 23:14:...
75537401,12600.0,1392.76,0.0,13992.757,3437.74,Aug-2018,null,2026-09-28 23:14:...
75038986,7805.03,7583.71,0.0,15388.74,440.72,Mar-2019,Apr-2019,2026-09-28 23:14:...
76301424,2614.78,1080.09,0.0,3694.87,153.89,Apr-2018,null,2026-09-28 23:14:...
75333198,8664.09,3431.32,68.8,12164.21,343.9,Mar-2019,Apr-2019,2026-09-28 23:14:...
76391453,35000.0,10266.01,0.0,45266.008,18884.35,Feb-2019,null,2026-09-28 23:14:...
76363364,15000.0,2093.11,0.0,17093.107,4648.59,Aug-2018,null,2026-09-28 23:14:...
76272510,14463.21,11178.79,110.13,25752.13,734.18,Mar-2019,Apr-2019,2026-09-28 23:14:...
76304116,4800.0,1051.65,0.0,5851.6523,1761.59,Dec-2017,null,2026-09-28 23:14:...


In [14]:
loans_payments_ndate_fixed_df = loans_payments_ldate_fixed_df.withColumn(
"next_payment_date",
when(
(col("next_payment_date") == 0),
None
).otherwise(col("next_payment_date"))
)

In [15]:
loans_payments_ndate_fixed_df

loan_id,total_principal_received,total_interest_received,total_late_fee_received,total_payment_received,last_payment_amount,last_payment_date,next_payment_date,Ingest_date
76003861,6423.8,5607.85,0.0,12031.65,574.88,Jan-2018,null,2026-09-29 01:10:...
76263914,2400.0,323.18,0.0,2723.1802,905.18,Dec-2017,null,2026-09-29 01:10:...
75537401,12600.0,1392.76,0.0,13992.757,3437.74,Aug-2018,null,2026-09-29 01:10:...
75038986,7805.03,7583.71,0.0,15388.74,440.72,Mar-2019,Apr-2019,2026-09-29 01:10:...
76301424,2614.78,1080.09,0.0,3694.87,153.89,Apr-2018,null,2026-09-29 01:10:...
75333198,8664.09,3431.32,68.8,12164.21,343.9,Mar-2019,Apr-2019,2026-09-29 01:10:...
76391453,35000.0,10266.01,0.0,45266.008,18884.35,Feb-2019,null,2026-09-29 01:10:...
76363364,15000.0,2093.11,0.0,17093.107,4648.59,Aug-2018,null,2026-09-29 01:10:...
76272510,14463.21,11178.79,110.13,25752.13,734.18,Mar-2019,Apr-2019,2026-09-29 01:10:...
76304116,4800.0,1051.65,0.0,5851.6523,1761.59,Dec-2017,null,2026-09-29 01:10:...


In [17]:
loans_payments_ndate_fixed_df.write \
.format("parquet") \
.mode("overwrite") \
.option("header",True) \
.option("path","/user/itv026931/lendingclubproject/cleaned/loan_repayment_parquet") \
.save()

In [18]:
loans_payments_ndate_fixed_df.write \
.format("csv") \
.mode("overwrite") \
.option("header",True) \
.option("path","/user/itv026931/lendingclubproject/cleaned/loan_repayment_csv") \
.save()